# Retrain Multi-Head Projections — Full Pipeline

| Source | Items | Compat Pairs | Signal |
|--------|-------|-------------|--------|
| **Polyvore** | 142K | 1.6M | outfit co-occurrence |
| **STL (Shop The Look)** | 33K | 881K | real e-commerce scenes |
| **FashionStylist** | 4.6K | 35K | expert-curated + attributes |

After merge with engagement weighting: **3M+ training pairs**

In [ ]:
import os, sys
ROOT = os.path.expanduser('~/loom')
os.chdir(ROOT)
sys.path.insert(0, ROOT)

import torch
device = 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Install dependencies

In [ ]:
!pip install -q datasets Pillow h5py httpx

## 2. Prepare data sources (already done, just verify)

In [ ]:
!python train/data/prepare_polyvore.py --data-dir data/polyvore --download
!python train/data/prepare_fashionstylist.py --output-dir data/fashionstylist
!python train/data/prepare_fashionrec.py --output-dir data/fashionrec

## 3. Merge all sources

In [ ]:
!python train/data/merge_datasets.py --output-dir data/merged

import os
for f in sorted(os.listdir('data/merged')):
    sz = os.path.getsize(f'data/merged/{f}')
    print(f'  {f}: {sz/1024/1024:.1f} MB')

## 4. Download Polyvore images

Original Polyvore CDN is dead (301 redirects to ssense.com HTML pages).
Uses Web Archive fallback via `train/download_images.py` which saves images as `{tid}.jpg`
matching the paths in `item_metadata.csv`.

In [ ]:
# Download via Web Archive (polyvore CDN is dead)
# Limit to 10K for training speed (142K available)
!python train/download_images.py --data-dir data/polyvore --limit 10000 --workers 16

import glob
n = len(glob.glob('data/polyvore/images/*.jpg'))
print(f'\nTotal Polyvore images on disk: {n}')

## 5. Pre-compute DINOv2 backbone embeddings

In [ ]:
!python train/precompute_backbones.py \
    --data-dir data/merged \
    --output data/merged/backbone_embeddings.h5 \
    --batch-size 32

In [ ]:
import h5py
with h5py.File('data/merged/backbone_embeddings.h5', 'r') as f:
    print(f'Embeddings: {f["embeddings"].shape}')
    print(f'Items with backbones: {f["item_ids"].shape[0]}')

## 6. Train all heads (with hard negative mining)

In [ ]:
!python train/train_heads.py --config train/config.yaml --head all

!ls -lh models/multihead/

## 7. Evaluate on Polyvore test set

In [ ]:
!python train/eval_ab_test.py --config train/config.yaml --data-dir data/polyvore

## 8. Backfill production DB + human eval

In [ ]:
!python scripts/backfill_multihead.py

In [ ]:
!python eval/human_eval_export.py --n-samples 20

import glob
latest = sorted(glob.glob('eval/results/human_eval_ab_*.html'))[-1]
print(f'\nOpen in browser: {latest}')